# **BankIntel AI Chatbot - BERT fine-tunning**

In [3]:
import pandas as pd
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

import numpy as np

### **Data Loading**

In [4]:
train_df = pd.read_csv("dataset/train.csv")
test_df = pd.read_csv("dataset/test.csv")
val_df = pd.read_csv("dataset/dev.csv")

In [5]:
print(train_df.shape)
train_df.sample(5)

(10003, 3)


,text,label,label_text
8622,The transfer I made isn't reflected in my balance,5,balance_not_updated_after_bank_transfer
5436,How do I get my money back for something I bou...,52,request_refund
6001,I transferred money to another country several...,48,pending_transfer
905,Can you tell me the names of the fiat currenci...,36,fiat_currency_support
436,the exchange rate at which my card payment is ...,17,card_payment_wrong_exchange_rate


In [6]:
print(test_df.shape)
test_df.sample(5)

(3080, 3)


,text,label,label_text
1932,The ATM machine stole my card.,18,card_swallowed
2674,Problems with top up,59,top_up_failed
95,Do you have a list of exchange rates?,32,exchange_rate
659,I topped up by card a while ago and it's still...,47,pending_top_up
1097,There is a card payment I do not recognise.,16,card_payment_not_recognised


In [7]:
print(val_df.shape)
val_df.sample(5)

(3080, 3)


,text,label,label_text
2243,Can I get my salary through this?,50,receiving_money
1479,locations to withdraw money,3,atm_support
650,I topped up but it didn't complete,47,pending_top_up
1357,Why don't I see my top up in my wallet?,62,topping_up_by_card
258,Will you handle EUR?,36,fiat_currency_support


### **Preprocessing**

In [8]:
train_df["label"].unique()

array([11, 13, 32, 17, 34, 46, 36, 12,  4, 14, 33, 41,  1, 49, 23, 56, 47,
        8, 60, 75, 15, 66, 54, 40, 10, 61,  6, 16, 30, 74, 68, 38, 73, 62,
       29, 22,  3, 28, 44, 26, 45, 42, 52, 27, 51, 25, 48, 55, 18, 63, 70,
       67, 53, 21,  7, 64, 50, 35, 65, 71, 39, 58, 43, 72, 76, 37, 59,  5,
       20, 31, 57,  0, 19,  9,  2, 69, 24])

In [30]:
label_df = pd.DataFrame({
    "label" : train_df['label'],
    "label_text" : train_df['label_text']
})

label_df_unique = label_df.drop_duplicates().sort_values("label")
label_map = dict(label_df_unique[['label', 'label_text']].values)

In [31]:
label_map

{0: 'activate_my_card',
 1: 'age_limit',
 2: 'apple_pay_or_google_pay',
 3: 'atm_support',
 4: 'automatic_top_up',
 5: 'balance_not_updated_after_bank_transfer',
 6: 'balance_not_updated_after_cheque_or_cash_deposit',
 7: 'beneficiary_not_allowed',
 8: 'cancel_transfer',
 9: 'card_about_to_expire',
 10: 'card_acceptance',
 11: 'card_arrival',
 12: 'card_delivery_estimate',
 13: 'card_linking',
 14: 'card_not_working',
 15: 'card_payment_fee_charged',
 16: 'card_payment_not_recognised',
 17: 'card_payment_wrong_exchange_rate',
 18: 'card_swallowed',
 19: 'cash_withdrawal_charge',
 20: 'cash_withdrawal_not_recognised',
 21: 'change_pin',
 22: 'compromised_card',
 23: 'contactless_not_working',
 24: 'country_support',
 25: 'declined_card_payment',
 26: 'declined_cash_withdrawal',
 27: 'declined_transfer',
 28: 'direct_debit_payment_not_recognised',
 29: 'disposable_card_limits',
 30: 'edit_personal_details',
 31: 'exchange_charge',
 32: 'exchange_rate',
 33: 'exchange_via_app',
 34: 'extr

In [32]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [33]:
# Test with single sentence
tokenizer("Hello bert model", return_tensors='pt', padding='max_length')

{'input_ids': tensor([[  101,  7592, 14324,  2944,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,  

In [34]:
def text_encoder(df):
    encodings  = tokenizer(
        df['text'].tolist(), # text list expects
        truncation=True,
        padding=True,
    )

    return encodings

train_encodings = text_encoder(train_df)
test_encodings = text_encoder(test_df)
val_encodings = text_encoder(val_df)

In [35]:
train_labels = train_df['label'].tolist()
train_labels[1:10]

test_labels = test_df['label'].tolist()
val_labels = val_df['label'].tolist()

In [36]:
print(type(train_encodings))
# print(train_encodings.keys())
print(len(train_encodings['input_ids']))
print(len(train_encodings['input_ids'][0]))
# print(train_encodings.items())

<class 'transformers.tokenization_utils_base.BatchEncoding'>
10003
98


In [37]:
for k, v in train_encodings.items():
    # print(len(encodings['input_ids'][0]))
    print(k, len(v))

input_ids 10003
token_type_ids 10003
attention_mask 10003


In [38]:
print(len(train_encodings['input_ids'][0]))
# train_encodings['input_ids'][0]
# train_encodings['token_type_ids'][0]
# train_encodings['attention_mask'][0]

98


### **Creating Dataset**
- Creating a custom dataset by extending PyTorch’s built-in Dataset class

In [39]:
type(train_encodings.items())

collections.abc.ItemsView

In [40]:
list(train_encodings.keys())

['input_ids', 'token_type_ids', 'attention_mask']

In [41]:
train_encodings["input_ids"][0]

[101,
 1045,
 2572,
 2145,
 3403,
 2006,
 2026,
 4003,
 1029,
 102,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [42]:
import torch

# converts: Text + labels → PyTorch format
class BankingDataset(torch.utils.data.Dataset):   # inheriting (extending) PyTorch’s Dataset class
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    # When training starts, PyTorch calls __getitem__ , Eg : train_ds[0]
    def __getitem__(self, idx):
        item = {key : torch.tensor(val[idx]) for key, val in self.encodings.items()} # take one sample, convert to tensors, create dictionary
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [43]:
# Creating obj of BankingDataset(in type pytorch Dataset), No tensor creation yet
train_ds = BankingDataset(train_encodings, train_labels)
test_ds = BankingDataset(test_encodings, test_labels) # final evaluation only
val_ds = BankingDataset(val_encodings, val_labels) # validation during training

In [44]:
type(train_ds)

__main__.BankingDataset

In [45]:
# For Hugging Face Trainer we must use 'labels' not 'label'
# if use pure PyTorch training loop since we can hanfle spelings does't mattter

train_ds[0]

{'input_ids': tensor([ 101, 1045, 2572, 2145, 3403, 2006, 2026, 4003, 1029,  102,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0]),
 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

### **Model**

In [48]:
# Craeting classifier model for bankin intent classification

model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(label_map.keys())
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [49]:
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

### **Define the params need to be trained**

In [50]:
def params_count(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
        "Total" : total_params,
        "Trainable" : trainable_params,
        "Frozen" : total_params - trainable_params
    }

In [51]:
# params_count(model)

In [52]:
# Freezing the model's (bert): BertModel(...) parameters
# still 768 x 77 +77 = 59,213 can ber changed that not in (bert) section

# NOTE : DUE TO LESS ACC DECIDED TO FINE TUNNE ENTIRE MODEL

# for param in model.base_model.parameters():
    # param.requires_grad = False

In [53]:
# params_count(model)

In [54]:
# Allow the pooler weights to be updated during training

# for param in model.bert.pooler.parameters():
    # param.requires_grad = True

In [55]:
params_count(model)

{'Total': 109541453, 'Trainable': 109541453, 'Frozen': 0}

In [56]:
# weights that can be changed while training

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

bert.embeddings.word_embeddings.weight
bert.embeddings.position_embeddings.weight
bert.embeddings.token_type_embeddings.weight
bert.embeddings.LayerNorm.weight
bert.embeddings.LayerNorm.bias
bert.encoder.layer.0.attention.self.query.weight
bert.encoder.layer.0.attention.self.query.bias
bert.encoder.layer.0.attention.self.key.weight
bert.encoder.layer.0.attention.self.key.bias
bert.encoder.layer.0.attention.self.value.weight
bert.encoder.layer.0.attention.self.value.bias
bert.encoder.layer.0.attention.output.dense.weight
bert.encoder.layer.0.attention.output.dense.bias
bert.encoder.layer.0.attention.output.LayerNorm.weight
bert.encoder.layer.0.attention.output.LayerNorm.bias
bert.encoder.layer.0.intermediate.dense.weight
bert.encoder.layer.0.intermediate.dense.bias
bert.encoder.layer.0.output.dense.weight
bert.encoder.layer.0.output.dense.bias
bert.encoder.layer.0.output.LayerNorm.weight
bert.encoder.layer.0.output.LayerNorm.bias
bert.encoder.layer.1.attention.self.query.weight
bert.enc

### Compute matrix function

In [57]:
acc_metric = evaluate.load('accuracy', download_mode='force_redownload')

def compute_metric(eval_pred):
    logits, labels = eval_pred   # logits - predicted class probs list by moel for val_ds, label - acutala label of those predicted labels

    predictions = np.argmax(logits, axis=1)   # Choose max vlaue from each logits list

    return acc_metric.compute(predictions=predictions, references=labels)

### Training arguments

In [58]:
training_args = TrainingArguments(
    output_dir='/result',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=10,

    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,

    warmup_steps=500,
    lr_scheduler_type='linear',

    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    report_to='none'
)

In [59]:
# from torch.optim import AdamW

# optimizer = AdamW(model.parameters(), lr=training_args.learning_rate, fused=False)

trainner = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metric,
    # optimizers=(optimizer, None)
)

In [60]:
trainner.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,3.866120,3.789322,0.251299
2,2.060575,1.836702,0.752922
3,1.104412,1.001817,0.848377
4,0.726495,0.709079,0.888961
5,0.561770,0.631676,0.895779


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=1565, training_loss=1.9629505355518084, metrics={'train_runtime': 952.7227, 'train_samples_per_second': 52.497, 'train_steps_per_second': 1.643, 'total_flos': 2520506592200100.0, 'train_loss': 1.9629505355518084, 'epoch': 5.0})

### Saved trained model from colab

In [62]:
import os

# dirs = [i for i in os.listdir("saved_models/") if i.startswith("v")]
# version = len( dirs) + 1
# print(dirs)
trainner.save_model(f"saved_models/v_1")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [63]:
import shutil

shutil.make_archive("model_v1", "zip", "saved_models/v_1")

'/content/model_v1.zip'

In [64]:
from google.colab import files
files.download("model_v1.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>